
# Module 2 — Voice Cloning

In [1]:

import sys
print("Python version:", sys.version)

major, minor = sys.version_info[:2]
if (major, minor) < (3, 10):
    print("❌ Python 3.10+ required. Please use a 3.10 or 3.11 environment.")
elif (major, minor) >= (3, 13):
    print("⚠️  Python 3.13+ may have compatibility issues. 3.11 is recommended.")
else:
    print("✅ Python version OK.")


Python version: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
✅ Python version OK.


In [2]:

# Check PyTorch and CUDA
try:
    import torch
    print("PyTorch version:", torch.__version__)

    if torch.cuda.is_available():
        print(f"✅ CUDA available: {torch.cuda.get_device_name(0)}")
        print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        if torch.cuda.get_device_properties(0).total_memory < 7e9:
            print("⚠️  Less than 7GB VRAM detected — VoxCPM2 may run slowly or OOM.")
    else:
        print("⚠️  No CUDA GPU detected.")
        print("   VoxCPM2 will run in CPU mode — very slow, but works for install verification.")
except ImportError:
    print("❌ PyTorch not installed. Run: pip install voxcpm  (installs PyTorch automatically)")


PyTorch version: 2.12.1+cu130
✅ CUDA available: NVIDIA GeForce RTX 5060 Ti
   VRAM: 16.6 GB


In [3]:

# Check VoxCPM itself
try:
    from voxcpm import VoxCPM
    print("✅ VoxCPM imported successfully.")
except ImportError as e:
    print(f"❌ VoxCPM not installed: {e}")
    print("   Run: pip install voxcpm")
except Exception as e:
    print(f"⚠️  VoxCPM import error: {e}")


❌ VoxCPM not installed: No module named 'voxcpm'
   Run: pip install voxcpm


In [4]:

import os
import json
import numpy as np
import soundfile as sf

print("All standard imports OK.")


All standard imports OK.



---
## 1. Load Module 1 Outputs

Point this notebook at the clean emotion clips produced by the Module 1 notebook.
The default paths below match the default output paths from Module 1 — change them
if you saved your clips somewhere else.


In [5]:

# ── POINT THESE AT YOUR MODULE 1 OUTPUTS ──────────────────────────────────────
# These are the cleaned emotion clip WAV files from Module 1's output/ folder.
# Change the base path if your Module 1 notebook is in a different location.

MODULE1_OUTPUT_DIR = "./live_demo/output/20260626_152440_8856ed"   # <-- adjust if needed

EMOTION_CLIPS = {
    "normal": f"{MODULE1_OUTPUT_DIR}/normal_clean.wav",
    "loud":   f"{MODULE1_OUTPUT_DIR}/loud_clean.wav",
    "happy":  f"{MODULE1_OUTPUT_DIR}/happy_clean.wav",
    "angry":  f"{MODULE1_OUTPUT_DIR}/angry_clean.wav",
    "sad":    f"{MODULE1_OUTPUT_DIR}/sad_clean.wav",
}

# The anchor selection from Module 1 Section 3.
# Update these if your Module 1 anchor selection chose different tags.
CALM_BASELINE_TAG = "normal"
EXPRESSIVE_PEAK_TAG = "loud"    # whichever had the highest RMS in Module 1


def check_clip(path):
    if not os.path.exists(path):
        return None
    y, sr = sf.read(path)
    duration = len(y) / sr
    rms = float(np.sqrt(np.mean(y.astype(np.float64) ** 2)))
    return {"duration": round(duration, 1), "sr": sr, "rms": round(rms, 4)}


print("Module 1 output clips:")
all_found = True
for tag, path in EMOTION_CLIPS.items():
    info = check_clip(path)
    if info:
        anchor = ""
        if tag == CALM_BASELINE_TAG:
            anchor = "  [CALM BASELINE ANCHOR]"
        elif tag == EXPRESSIVE_PEAK_TAG:
            anchor = "  [EXPRESSIVE PEAK ANCHOR]"
        print(f"  ✅ [{tag:>6}]  {info['duration']}s  rms={info['rms']}{anchor}")
    else:
        print(f"  ⚠️  [{tag:>6}]  Not found at '{path}'")
        all_found = False

if not all_found:
    print()
    print("Some clips not found — run Module 1 notebook first, or adjust MODULE1_OUTPUT_DIR above.")


Module 1 output clips:
  ✅ [normal]  7.0s  rms=0.1144  [CALM BASELINE ANCHOR]
  ✅ [  loud]  7.0s  rms=0.1208  [EXPRESSIVE PEAK ANCHOR]
  ✅ [ happy]  7.0s  rms=0.0727
  ✅ [ angry]  7.0s  rms=0.0699
  ✅ [   sad]  7.0s  rms=0.1255



---
## 2. Basic Generation Test

Before trying to clone anyone's voice, first confirm that VoxCPM2 runs at all on
your machine. This cell loads the model and generates one sentence with no reference
audio (pure text-to-speech, no cloning yet). If this works, you know the model and
your GPU/CPU are cooperating correctly.

> **First run only:** the model weights (~5GB) download automatically from Hugging Face
> the first time you run this. This takes a few minutes depending on your connection.
> Every run after that is instant (weights are cached locally).


In [6]:
if 'model' in globals():
    print("Active model detected in memory.")
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated(0) / 1e9
        print(f"  Current VRAM allocated: {vram_before:.2f} GB")

In [9]:
if torch.cuda.is_available():
    vram_before = torch.cuda.memory_allocated(0) / 1e9
    print(f"  Current VRAM allocated: {vram_before:.2f} GB")

  Current VRAM allocated: 5.31 GB


In [8]:

import torch
from voxcpm import VoxCPM
import soundfile as sf
import time

# Detect device automatically — GPU if available, CPU otherwise
device = "cuda" if torch.cuda.is_available() else "cpu"
# Set optimize=False to disable torch.compile, which is extremely RAM-intensive
# and can freeze 16GB RAM systems during compilation of this 2B parameter model.
optimize = False

print(f"Loading VoxCPM2 on device: {device}")
print("(First run downloads ~5GB of weights — this may take a few minutes.)")

t0 = time.time()
model = VoxCPM.from_pretrained(
    "openbmb/VoxCPM2",
    load_denoiser=True,   # denoiser not needed here; we already cleaned audio in Module 1
    optimize=optimize,
)
print(f"Model loaded in {time.time()-t0:.1f}s")


Loading VoxCPM2 on device: cuda
(First run downloads ~5GB of weights — this may take a few minutes.)


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

voxcpm_model_path: /home/riad/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/bffb3df5a29440629464e5e839f4d214c8714c3d, zipenhancer_model_path: iic/speech_zipenhancer_ans_multiloss_16k_base, enable_denoiser: True
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'VoxCPM2Tokenizer'. 
The class this function is called from is 'LlamaTokenizerFast'.
/home/riad/Documents/Pronob/venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Loading AudioVAE from pytorch: /home/riad/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/bffb3df5a29440629464e5e839f4d214c8714c3d/audiovae.pth
Running on device: cuda, dtype: bfloat16
Loading model from safetensors: /ho

2026-06-26 15:49:29,498 - modelscope - INFO - initiate model from /home/riad/.cache/modelscope/hub/models/iic/speech_zipenhancer_ans_multiloss_16k_base
2026-06-26 15:49:29,498 - modelscope - INFO - initiate model from location /home/riad/.cache/modelscope/hub/models/iic/speech_zipenhancer_ans_multiloss_16k_base.
2026-06-26 15:49:29,499 - modelscope - INFO - initialize model from /home/riad/.cache/modelscope/hub/models/iic/speech_zipenhancer_ans_multiloss_16k_base
2026-06-26 15:49:29,499 - modelscope - WARNING - Use allow_remote=True. Will invoke codes from /home/riad/.cache/modelscope/hub/models/iic/speech_zipenhancer_ans_multiloss_16k_base. Please make sure that you can trust the external codes.
2026-06-26 15:49:29,627 - modelscope - WARNING - No preprocessor field found in cfg.
2026-06-26 15:49:29,627 - modelscope - WARNING - No val key and type key found in preprocessor domain of configuration.json file.
2026-06-26 15:49:29,628 - modelscope - WARNING - Cannot find available config t

Model loaded in 29.1s


In [10]:

# Basic generation — no reference audio, just text-to-speech
TEST_SENTENCE = "The weather today is absolutely wonderful, and I can't wait to go outside."

print(f"Generating: '{TEST_SENTENCE}'")
t0 = time.time()

wav = model.generate(
    text=TEST_SENTENCE,
    cfg_value=2.0,
    inference_timesteps=10,
)

elapsed = time.time() - t0
sr = model.tts_model.sample_rate

os.makedirs("output", exist_ok=True)
output_path = "output/test_basic_generation.wav"
sf.write(output_path, wav, sr)

print(f"✅ Generated in {elapsed:.1f}s — saved to {output_path}")
print(f"   Audio length: {len(wav)/sr:.1f}s · Sample rate: {sr}Hz")
print(f"   Real-time factor: {elapsed / (len(wav)/sr):.2f}x",
      "(< 1.0 = faster than real-time)" if elapsed < len(wav)/sr else "(> 1.0 = slower than real-time)")


Generating: 'The weather today is absolutely wonderful, and I can't wait to go outside.'


 24%|██▍       | 27/112 [00:02<00:07, 11.28it/s]


✅ Generated in 3.5s — saved to output/test_basic_generation.wav
   Audio length: 4.5s · Sample rate: 48000Hz
   Real-time factor: 0.78x (< 1.0 = faster than real-time)



> **What to listen for.** Play `output/test_basic_generation.wav` — it should be clear,
> natural-sounding speech in a generic voice. This only confirms the model runs; the
> actual cloning quality test is in Section 3.



---
## 3. Voice Cloning Test

Now the real test: use the **calm baseline anchor clip** from Module 1 as the reference,
and generate a new sentence in that cloned voice. This is the core capability — does the
output sound like the same person who recorded the enrollment video?


In [11]:

# Path to the calm baseline anchor clip from Module 1
CALM_ANCHOR_PATH = EMOTION_CLIPS[CALM_BASELINE_TAG]

if not os.path.exists(CALM_ANCHOR_PATH):
    print(f"⚠️  Calm anchor clip not found at '{CALM_ANCHOR_PATH}'.")
    print("    Run Module 1 first, then adjust MODULE1_OUTPUT_DIR in Section 1.")
else:
    print(f"Reference clip: {CALM_ANCHOR_PATH}")
    info = check_clip(CALM_ANCHOR_PATH)
    print(f"  Duration: {info['duration']}s · RMS: {info['rms']}")


Reference clip: ./live_demo/output/20260626_152440_8856ed/normal_clean.wav
  Duration: 7.0s · RMS: 0.1144


In [ ]:
CLONE_TEST_SENTENCE="Effective system monitoring is an essential skill for maintaining a stable and responsive computing environment, especially when working with resource-intensive applications such as modern IDEs or AI tools. In operating systems like Ubuntu, performance issues often arise not from CPU overload but from excessive memory consumption. When applications continue running background processes even after being closed, they may gradually consume more RAM, eventually leading to system slowdowns or complete freezing. This situation is commonly associated with memory leaks or improperly terminated child processes. Tools like `htop`, `ps`, and `free` provide real-time insights into system resource usage, allowing users to identify which processes are consuming the most memory or CPU. Understanding how to interpret these metrics is crucial for diagnosing problems accurately. Additionally, managing the clipboard effectively and ensuring proper application behavior can significantly improve productivity. By adopting good practices—such as monitoring processes before launching heavy applications, limiting concurrent resource-heavy tools, and terminating unnecessary background tasks—users can maintain system stability. Ultimately, a disciplined approach to system observation and process management enables developers and power users to prevent performance bottlenecks and ensure a smoother workflow."

In [ ]:
import re
import time
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

# --- 1. Text Segmentation ---
# Objectives: Breaks down long text to prevent sequence length attention drift.
# Prevents VoxCPM2 from generating static noise and hallucinations after ~15-20 seconds.
sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', CLONE_TEST_SENTENCE) if s.strip()]
print(f"Segmented paragraph into {len(sentences)} sentences.")

all_segments = []
sample_rate = model.tts_model.sample_rate

# Track total generation time
total_generation_time = 0.0

# --- 2. Sentence-by-Sentence Generation Loop ---
for i, sentence in enumerate(sentences):
    print(f"Generating sentence {i+1}/{len(sentences)}: '{sentence[:40]}...'")
    try:
        t_start = time.time()
        
        segment = model.generate(
            text=sentence,
            reference_wav_path=MODULE1_OUTPUT_DIR,
            
            # cfg_value=2.0 (Classifier-Free Guidance): 
            # Controls similarity and stability of the cloned voice. 
            # Higher values increase speaker similarity but can introduce distortion. 
            # 2.0 is the optimal baseline.
            cfg_value=2.0,
            
            # inference_timesteps=10 (Denoising Steps):
            # Controls the speed-to-quality ratio. 
            # Lower value (e.g., 10) ensures very fast inference (lower RTF). 
            # Higher values (e.g., 20-30) produce studio quality but slow down generation.
            inference_timesteps=10,
        )
        
        total_generation_time += (time.time() - t_start)
        all_segments.append(segment)
    except Exception as e:
        print(f"Error generating sentence {i+1}: {e}")

# --- 3. Concatenation and Comfort Silence ---
if all_segments:
    # silence_duration=0.25:
    # Adds a natural 250ms pause between sentences to simulate breathing/speaking rhythm.
    silence_duration = 0.25  
    silence_samples = np.zeros(int(sample_rate * silence_duration))
    
    final_audio_list = []
    for idx, seg in enumerate(all_segments):
        final_audio_list.append(seg)
        if idx < len(all_segments) - 1:
            final_audio_list.append(silence_samples)
            
    wav_cloned = np.concatenate(final_audio_list, axis=0)
    
    # --- 4. Performance Metrics (RTF) ---
    # Real-Time Factor (RTF) = total inference duration / total audio duration.
    # An RTF < 1.0 means generation was faster than real-time playback.
    total_audio_duration = len(wav_cloned) / sample_rate
    rtf = total_generation_time / total_audio_duration
    
    print("\n=== PERFORMANCE METRICS ===")
    print(f"Inference Time: {total_generation_time:.2f} seconds")
    print(f"Audio Duration: {total_audio_duration:.2f} seconds")
    print(f"Real-Time Factor (RTF): {rtf:.3f}x ({'Faster' if rtf < 1 else 'Slower'} than real-time)")
    
    # --- 5. Save & Play ---
    output_path = "output/test_cloned_voice_2.wav"
    sf.write(output_path, wav_cloned, sample_rate)
    print(f"\nSaved clean file to: {output_path}")
    
    # Display the player to listen immediately
    display(Audio(wav_cloned, rate=sample_rate))
else:
    print("❌ Failed to generate any segments.")


In [ ]:
# CLONE_TEST_SENTENCE="Effective system monitoring is an essential skill for maintaining a stable and responsive computing environment, especially when working with resource-intensive applications such as modern IDEs or AI tools. In operating systems like Ubuntu, performance issues often arise not from CPU overload but from excessive memory consumption. When applications continue running background processes even after being closed, they may gradually consume more RAM, eventually leading to system slowdowns or complete freezing. This situation is commonly associated with memory leaks or improperly terminated child processes. Tools like `htop`, `ps`, and `free` provide real-time insights into system resource usage, allowing users to identify which processes are consuming the most memory or CPU. Understanding how to interpret these metrics is crucial for diagnosing problems accurately. Additionally, managing the clipboard effectively and ensuring proper application behavior can significantly improve productivity. By adopting good practices—such as monitoring processes before launching heavy applications, limiting concurrent resource-heavy tools, and terminating unnecessary background tasks—users can maintain system stability. Ultimately, a disciplined approach to system observation and process management enables developers and power users to prevent performance bottlenecks and ensure a smoother workflow."

# # Clone the voice using the calm anchor clip as reference
# #CLONE_TEST_SENTENCE = "Hello. This is my voice, generated by a cloning model."

# if os.path.exists(CALM_ANCHOR_PATH):
#     print(f"Cloning voice from: {CALM_ANCHOR_PATH}")
#     print(f"Generating: '{CLONE_TEST_SENTENCE}'")

#     t0 = time.time()
#     wav_cloned = model.generate(
#         text=CLONE_TEST_SENTENCE,
#         reference_wav_path=CALM_ANCHOR_PATH,
#         cfg_value=2.0,
#         inference_timesteps=10,
#     )
#     elapsed = time.time() - t0

#     output_path = "output/test_cloned_voice.wav"
#     sf.write(output_path, wav_cloned, model.tts_model.sample_rate)

#     print(f"✅ Cloned voice generated in {elapsed:.1f}s — saved to {output_path}")
#     print(f"   Audio length: {len(wav_cloned)/model.tts_model.sample_rate:.1f}s")
# else:
#     print("Skipped — reference clip not found.")


Cloning voice from: ./live_demo/output/20260626_152440_8856ed/normal_clean.wav
Generating: 'Effective system monitoring is an essential skill for maintaining a stable and responsive computing environment, especially when working with resource-intensive applications such as modern IDEs or AI tools. In operating systems like Ubuntu, performance issues often arise not from CPU overload but from excessive memory consumption. When applications continue running background processes even after being closed, they may gradually consume more RAM, eventually leading to system slowdowns or complete freezing. This situation is commonly associated with memory leaks or improperly terminated child processes. Tools like `htop`, `ps`, and `free` provide real-time insights into system resource usage, allowing users to identify which processes are consuming the most memory or CPU. Understanding how to interpret these metrics is crucial for diagnosing problems accurately. Additionally, managing the clipboa

 31%|███       | 435/1402 [00:35<01:19, 12.12it/s]


✅ Cloned voice generated in 37.8s — saved to output/test_cloned_voice.wav
   Audio length: 69.8s



> **What to listen for.** Play `output/test_cloned_voice.wav` and compare it to
> your original `normal_clean.wav`. The pitch, timbre, and speaking rhythm should
> be recognisably similar — the model is learning *how you sound*, not what you said.
> Don't expect perfection from a short reference clip; look for whether the basic
> voice character carries through.



---
## 4. Controllable Emotion Cloning

This is the key capability for the audiobook product. VoxCPM2 supports **style guidance
via parenthetical instructions** at the start of the text — for example, `(slightly faster,
cheerful)` — which allows generating the same voice at different emotional intensities
without re-recording.

This section generates the same test sentence at three points on the emotional scale:
calm (matching the baseline anchor), expressive (matching the peak anchor), and a
mid-point between them — all from the same calm reference clip.


In [ ]:

BOOK_SENTENCE = "She opened the door slowly, wondering what she might find on the other side."

# Three generations from the same reference — only the style instruction changes
EMOTION_VARIATIONS = [
    ("calm",       ""),
    ("warm",       "(warm, gentle, slightly slower)"),
    ("expressive", "(energetic, expressive, clear emphasis)"),
]

if os.path.exists(CALM_ANCHOR_PATH):
    for label, style_prefix in EMOTION_VARIATIONS:
        full_text = f"{style_prefix}{BOOK_SENTENCE}" if style_prefix else BOOK_SENTENCE

        print(f"Generating [{label}]...")
        t0 = time.time()
        wav_out = model.generate(
            text=full_text,
            reference_wav_path=CALM_ANCHOR_PATH,
            cfg_value=2.0,
            inference_timesteps=10,
        )
        elapsed = time.time() - t0

        output_path = f"output/emotion_{label}.wav"
        sf.write(output_path, wav_out, model.tts_model.sample_rate)
        print(f"  ✅ Saved to {output_path}  ({elapsed:.1f}s)")

    print()
    print("Listen to all three and compare.")
    print("  calm       -> output/emotion_calm.wav")
    print("  warm       -> output/emotion_warm.wav")
    print("  expressive -> output/emotion_expressive.wav")
else:
    print("Skipped — reference clip not found.")



> **What to listen for.**
> - Does the voice identity (timbre, pitch character) stay consistent across the three outputs?
> - Does the emotional delivery actually shift — is the expressive version noticeably more
>   energetic than the calm one, while still sounding like the same person?
>
> These two questions together determine whether VoxCPM2 is ready to use as Module 2's
> cloning engine, or whether a different model (OmniVoice, OpenVoice) should be evaluated
> next. Document your findings in the R&D note cell below.

**R&D note — fill this in after listening:**


In [ ]:

# Fill in your observations after listening to the outputs above.
# This becomes part of the model selection decision for Sub-module 2A.

RND_OBSERVATIONS = {
    "voice_identity_preserved": None,    # True / False / "partially"
    "emotion_control_works":    None,    # True / False / "partially"
    "generation_speed_rtf":     None,    # e.g. 0.3 (fill in from Section 3 output)
    "audio_quality":            None,    # "good" / "acceptable" / "poor"
    "notes":                    "",      # anything else you noticed
}

print("RND_OBSERVATIONS (fill these in after listening):")
for k, v in RND_OBSERVATIONS.items():
    print(f"  {k}: {v}")



---
## 5. Save the Voice Profile

For VoxCPM2, the "voice profile" is simply the reference audio clip itself — the model
does zero-shot cloning, meaning it reads the reference at generation time and doesn't need
a pre-computed embedding stored separately. This is simpler than some other models, but
it means the reference WAV file **is** the profile and must be stored per user.

This section saves a clean, structured profile package that Module 3 can later load
directly — a folder with the reference audio plus a small JSON metadata file.


In [ ]:

import shutil

VOICE_PROFILE_DIR = "output/voice_profile"
os.makedirs(VOICE_PROFILE_DIR, exist_ok=True)

if os.path.exists(CALM_ANCHOR_PATH):
    # Copy the calm anchor as the primary reference audio
    profile_audio_path = f"{VOICE_PROFILE_DIR}/reference.wav"
    shutil.copy2(CALM_ANCHOR_PATH, profile_audio_path)

    # Also copy the expressive peak anchor — available for advanced use later
    peak_path = EMOTION_CLIPS[EXPRESSIVE_PEAK_TAG]
    if os.path.exists(peak_path):
        shutil.copy2(peak_path, f"{VOICE_PROFILE_DIR}/reference_expressive.wav")

    # Write a metadata file so Module 3 knows how to use this profile
    profile_metadata = {
        "model": "VoxCPM2",
        "reference_audio": "reference.wav",
        "reference_expressive_audio": "reference_expressive.wav",
        "calm_baseline_tag": CALM_BASELINE_TAG,
        "expressive_peak_tag": EXPRESSIVE_PEAK_TAG,
        "cloning_mode": "zero_shot",
        "recommended_cfg_value": 2.0,
        "recommended_inference_timesteps": 10,
        "notes": "Reference audio from Module 1 enrollment recording.",
    }
    with open(f"{VOICE_PROFILE_DIR}/profile.json", "w") as f:
        json.dump(profile_metadata, f, indent=2)

    print(f"✅ Voice profile saved to: {VOICE_PROFILE_DIR}/")
    print(f"   Files:")
    for fname in os.listdir(VOICE_PROFILE_DIR):
        size = os.path.getsize(f"{VOICE_PROFILE_DIR}/{fname}")
        print(f"     {fname}  ({size/1024:.0f} KB)")
else:
    print("Skipped — anchor clips not found. Run Module 1 first.")



---
## 6. Combined Result — What Module 3 Would Receive

This is the shape of the data this module hands off once the voice profile is saved and
generation is confirmed working. Module 3 (Audiobook Management) will load the profile
and call `model.generate(text=..., reference_wav_path=profile["reference_audio"])` for
every chapter a user requests.


In [ ]:

def build_module2_result(profile_dir, rnd_observations, model_name="VoxCPM2"):
    profile_json_path = f"{profile_dir}/profile.json"
    profile_exists = os.path.exists(profile_json_path)

    generation_confirmed = (
        os.path.exists("output/test_cloned_voice.wav") and
        os.path.exists("output/emotion_calm.wav")
    )

    return {
        "module": "module_2_voice_cloning",
        "model_used": model_name,
        "overall_result": "pass" if (profile_exists and generation_confirmed) else "fail",
        "voice_profile_location": profile_dir if profile_exists else None,
        "generation_confirmed": generation_confirmed,
        "rnd_observations": rnd_observations,
        "handoff_to_module_3": {
            "load_profile_from": profile_dir,
            "generate_call": "model.generate(text=..., reference_wav_path=profile['reference_audio'])",
        },
    }


result = build_module2_result(
    VOICE_PROFILE_DIR,
    RND_OBSERVATIONS if 'RND_OBSERVATIONS' in dir() else {},
)
print(json.dumps(result, indent=2))



---
## Next Steps

1. **Listen to `output/test_cloned_voice.wav`** — does it sound like you?
2. **Listen to `output/emotion_calm.wav` vs `output/emotion_expressive.wav`** — does
   emotional control work while keeping your voice identity consistent?
3. **Fill in `RND_OBSERVATIONS`** in Section 4 with your honest assessment.
4. **If results are acceptable:** Module 2 is proven — proceed to wiring into FastAPI.
5. **If results need improvement:** try the same test with the expressive peak anchor
   (`loud_clean.wav`) as the reference instead of the calm one. Some voices clone better
   from a more energetic sample. Then consider evaluating OpenVoice or OmniVoice as
   alternatives (Sub-module 2A model comparison).
6. **Only once satisfied:** wrap the `model.generate()` call into a FastAPI endpoint
   per the "script first, FastAPI second" rule.
